<a href="https://colab.research.google.com/github/Salma19mostafa/Salma19mostafa/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow scikit-learn pandas matplotlib seaborn


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau


In [ ]:
# Load dataset
data = pd.read_csv("creditcard.csv")

# Handle missing values
data = data.fillna(data.mean())

# Encode categorical columns if needed
for col in data.select_dtypes(include=['object']).columns:
    data[col] = LabelEncoder().fit_transform(data[col])

# Features & target
X = data.drop("target_column", axis=1).values
y = data["target_column"].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.4),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # Change to softmax for multi-class, linear for regression
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint("best_model.h5", save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]


In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks
)


In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.2f}")

# Predictions
y_pred = (model.predict(X_test) > 0.5).astype("int32")

print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# Training curves
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend()
plt.show()


In [ ]:
reg_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.4),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')  # Linear output for regression
])

reg_model.compile(optimizer='adam', loss='mse', metrics=['mae'])


In [ ]:
reg_history = reg_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks
)


In [ ]:
loss, mae = reg_model.evaluate(X_test, y_test)
print(f"Test MAE: {mae:.2f}")

# Predictions
y_pred = reg_model.predict(X_test)

# Calculate R²
from sklearn.metrics import r2_score
print(f"R² Score: {r2_score(y_test, y_pred):.2f}")


In [ ]:
plt.plot(reg_history.history['loss'], label='train loss')
plt.plot(reg_history.history['val_loss'], label='val loss')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()


In [ ]:
target = "target_column"

# Check target type
y = data[target].values

if pd.api.types.is_numeric_dtype(data[target]):
    # If numeric but few unique values, treat as classification
    if len(np.unique(y)) < 20:
        task_type = "classification"
    else:
        task_type = "regression"
else:
    task_type = "classification"

print("Detected task type:", task_type)


In [ ]:
def build_model(task_type, input_dim, num_classes=None):
    model = Sequential([
        Dense(128, activation='relu', input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.4),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu')
    ])

    if task_type == "classification":
        if num_classes == 2:
            model.add(Dense(1, activation='sigmoid'))
            model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        else:
            model.add(Dense(num_classes, activation='softmax'))
            model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    else:  # regression
        model.add(Dense(1, activation='linear'))
        model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    return model


In [ ]:
if task_type == "classification":
    num_classes = len(np.unique(y))
    model = build_model(task_type, X_train.shape[1], num_classes)
else:
    model = build_model(task_type, X_train.shape[1])

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks
)

# Evaluation
if task_type == "classification":
    loss, acc = model.evaluate(X_test, y_test)
    print(f"Test Accuracy: {acc:.2f}")
else:
    loss, mae = model.evaluate(X_test, y_test)
    print(f"Test MAE: {mae:.2f}")


In [ ]:
if task_type == "classification":
    y_pred = (model.predict(X_test) > 0.5).astype("int32") if num_classes == 2 else np.argmax(model.predict(X_test), axis=1)

    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.show()


In [ ]:
if task_type == "regression":
    y_pred = model.predict(X_test)

    # Scatter plot
    plt.figure(figsize=(6,4))
    plt.scatter(y_test, y_pred, alpha=0.6)
    plt.xlabel("Actual Values")
    plt.ylabel("Predicted Values")
    plt.title("Predicted vs Actual")
    plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red')  # perfect fit line
    plt.show()

    # Residuals
    residuals = y_test - y_pred.flatten()
    plt.figure(figsize=(6,4))
    plt.hist(residuals, bins=30, edgecolor='black')
    plt.xlabel("Residuals")
    plt.ylabel("Frequency")
    plt.title("Residuals Distribution")
    plt.show()


In [ ]:
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title("Training vs Validation Loss")
plt.show()


In [ ]:
def run_pipeline(data, target_column, epochs=50, batch_size=32):
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.metrics import classification_report, confusion_matrix, r2_score
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

    # --- Preprocessing ---
    df = data.copy()
    df = df.fillna(df.mean())
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = LabelEncoder().fit_transform(df[col])

    X = df.drop(target_column, axis=1).values
    y = df[target_column].values

    # Detect task type
    if pd.api.types.is_numeric_dtype(df[target_column]):
        task_type = "classification" if len(np.unique(y)) < 20 else "regression"
    else:
        task_type = "classification"
    print("Detected task type:", task_type)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # --- Build Model ---
    model = Sequential([
        Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        BatchNormalization(),
        Dropout(0.4),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu')
    ])

    if task_type == "classification":
        num_classes = len(np.unique(y))
        if num_classes == 2:
            model.add(Dense(1, activation='sigmoid'))
            model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        else:
            model.add(Dense(num_classes, activation='softmax'))
            model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    else:
        model.add(Dense(1, activation='linear'))
        model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    # --- Callbacks ---
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ModelCheckpoint("best_model.h5", save_best_only=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
    ]

    # --- Train ---
    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
        callbacks=callbacks,
        verbose=0
    )

    # --- Evaluate ---
    if task_type == "classification":
        loss, acc = model.evaluate(X_test, y_test, verbose=0)
        print(f"Test Accuracy: {acc:.2f}")
        y_pred = (model.predict(X_test) > 0.5).astype("int32") if num_classes == 2 else np.argmax(model.predict(X_test), axis=1)
        print(classification_report(y_test, y_pred))
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title("Confusion Matrix")
        plt.show()
    else:
        loss, mae = model.evaluate(X_test, y_test, verbose=0)
        print(f"Test MAE: {mae:.2f}")
        y_pred = model.predict(X_test)
        print(f"R² Score: {r2_score(y_test, y_pred):.2f}")
        plt.scatter(y_test, y_pred, alpha=0.6)
        plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red')
        plt.xlabel("Actual")
        plt.ylabel("Predicted")
        plt.title("Predicted vs Actual")
        plt.show()
        residuals = y_test - y_pred.flatten()
        plt.hist(residuals, bins=30, edgecolor='black')
        plt.title("Residuals Distribution")
        plt.show()

    # --- Training Curves ---
    plt.plot(history.history['loss'], label='train loss')
    plt.plot(history.history['val_loss'], label='val loss')
    plt.legend()
    plt.title("Training vs Validation Loss")
    plt.show()

    return model


In [ ]:
# Example usage
model = run_pipeline(data, "target_column")
